In [19]:
import os 
import time 
import numpy as np
import pandas as pd
from iminuit import Minuit
from scipy.special import j0
import plotly.graph_objects as go
from iminuit.cost import LeastSquares
from scipy.integrate import fixed_quad
import joblib

In [20]:
save_folder = 'run7'
n_points = 10000

lower_factor = 0.99
upper_factor = 2 - lower_factor

In [21]:
# Load experimental data
atlas_data = pd.read_csv('../../../data/ens_atlas_difc0_2.dat', delim_whitespace=True, header=None)
totem_data = pd.read_csv('../../../data/ens_totem_difc0_2.dat', delim_whitespace=True, header=None)

# Function to process data for each experiment
def process_data(data, energy_blocks):
    x_values = []
    y_values = []
    y_errors = []
    
    for start, end in energy_blocks:
        block = data.iloc[start:end] if end is not None else data.iloc[start:]
        x_values.append(block[0].values)
        y_values.append(block[1].values)
        y_errors.append(block[2].values)
    
    return x_values, y_values, y_errors

# Energy ranges for each experiment (7TeV, 8TeV, 13TeV)
atlas_blocks = [(0, 29), (29, 58), (58, None)]
totem_blocks = [(0, 65), (65, 118), (118, None)]

# Process data
x_atlas, y_atlas, yerr_atlas = process_data(atlas_data, atlas_blocks)

# Extract values by energy (index 0=7TeV, 1=8TeV, 2=13TeV)
x_7_atlas, y_7_atlas, yerr_7_atlas = x_atlas[0], y_atlas[0], yerr_atlas[0]
x_8_atlas, y_8_atlas, yerr_8_atlas = x_atlas[1], y_atlas[1], yerr_atlas[1]
x_13_atlas, y_13_atlas, yerr_13_atlas = x_atlas[2], y_atlas[2], yerr_atlas[2]

/tmp/ipykernel_7463/4028186587.py:2: FutureWarning: The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead
  atlas_data = pd.read_csv('../../../data/ens_atlas_difc0_2.dat', delim_whitespace=True, header=None)
/tmp/ipykernel_7463/4028186587.py:3: FutureWarning: The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead
  totem_data = pd.read_csv('../../../data/ens_totem_difc0_2.dat', delim_whitespace=True, header=None)


In [22]:

b_0 = (33 - 6) / (12 * np.pi)
Lambda = 0.284  # ΛQCD in GeV
gamma_1 = 0.084
gamma_2 = 2.36
rho = 4.0
s0 = 1.0
alpha_prime = 0.25



ensemble_parameters = {
    'atlas': {
        'log': {
            'epsilon': 0.0753,
            'mg': 0.356,
            'a1': 1.373,
            'a2': 2.50
        },
        'pl': {
            'epsilon': 0.0753,
            'mg': 0.421,
            'a1': 1.517,
            'a2': 2.05
        }
    },
    'totem': {
        'log': {
            'epsilon': 0.0892,
            'mg': 0.380,
            'a1': 1.491,
            'a2': 2.77
        },
        'pl':{
            'epsilon': 0.0892,
            'mg': 0.447,
            'a1': 1.689,
            'a2': 1.7
        }
    }
}

ensemble_atlas = 'atlas'  
ensemble_totem = 'totem'

log_model_type = 'log'
pl_model_type = 'pl'   

def get_parameters_with_variations(ensemble_parameters, ensemble_name, model_type, lower_factor=lower_factor, upper_factor=upper_factor):
    # Obtém os parâmetros iniciais
    initial_params = ensemble_parameters[ensemble_name][model_type]
    
    # Cria as variações
    initial_params_low = {k: v * lower_factor for k, v in initial_params.items()}
    initial_params_high = {k: v * upper_factor for k, v in initial_params.items()}
    
    return initial_params, initial_params_low, initial_params_high

# Get parameters for selected configuration
initial_params_pl_atlas = ensemble_parameters[ensemble_atlas][pl_model_type]

# Para Atlas
initial_params_pl_atlas, initial_params_low_pl_atlas, initial_params_high_pl_atlas = \
    get_parameters_with_variations(ensemble_parameters, ensemble_atlas, pl_model_type)




In [23]:
def m2_log(q2, mg):
    lambda_squared = Lambda ** 2
    rho_mg_squared = rho * mg ** 2
    ratio = np.log((q2 + rho_mg_squared) / lambda_squared) / np.log(rho_mg_squared / lambda_squared)
    return mg ** 2 * ratio ** (-1 - gamma_1)

def m2_pl(q2, mg):
    lambda_squared = Lambda ** 2
    rho_mg_squared = rho * mg ** 2
    ratio = np.log((q2 + rho_mg_squared) / lambda_squared) / np.log(rho_mg_squared / lambda_squared)
    return (mg ** 4 / (q2 + mg ** 2)) * ratio ** (gamma_2 - 1)


def G_p(q2, a1, a2):
    return np.exp(-(a1 * q2 + a2 * q2 ** 2))

def alpha_D(q2, mg, m2_func):
    m2 = m2_func(q2, mg)
    return 1.0 / (b_0 * (q2 + m2) * np.log((q2 + 4 * m2) / (Lambda ** 2)))

def T_1(k, q, phi, mg, a1, a2, m2_func):
    q2 = q 
    qk_cos = np.sqrt(q) * k * np.cos(phi)
    qk_plus_squared = q2 / 4 + qk_cos + k ** 2
    qk_minus_squared = q2 / 4 - qk_cos + k ** 2
    alpha_D_plus = alpha_D(qk_plus_squared, mg, m2_func)
    alpha_D_minus = alpha_D(qk_minus_squared, mg, m2_func)
    G0 = G_p(q2, a1, a2)
    return alpha_D_plus * alpha_D_minus * G0 ** 2

def T_2(k, q, phi, mg, a1, a2, m2_func):
    q2 = q 
    qk_cos = np.sqrt(q) * k * np.cos(phi)
    qk_plus_squared = q2 / 4 + qk_cos + k ** 2
    qk_minus_squared = q2 / 4 - qk_cos + k ** 2
    alpha_D_plus = alpha_D(qk_plus_squared, mg, m2_func)
    alpha_D_minus = alpha_D(qk_minus_squared, mg, m2_func)
    factor = q2 + 9 * abs(k ** 2 - q2 / 4)
    G0 = G_p(q2, a1, a2)
    G_minus = G_p(factor, a1, a2)
    return alpha_D_plus * alpha_D_minus * G_minus * (2 * G0 - G_minus)

def integrand(y, x, mg, a1, a2, m2_func, q_val, sqrt_s):
    k = sqrt_s * x 
    phi = 2 * np.pi * y
    jacobian = 2 * np.pi * sqrt_s 
    return k * (T_1(k, q_val, phi, mg, a1, a2, m2_func) - T_2(k, q_val, phi, mg, a1, a2, m2_func)) * jacobian 

def sigma_tot(amp_value, s):
    return amp_value.imag / s * 0.389379323

def amp_calculation(diff_T, s, epsilon, t):
    alpha_pomeron = 1.0 + epsilon + alpha_prime * t
    regge_factor = (s**alpha_pomeron) * 1/(s0**(alpha_pomeron-1))
    return 1j * 8 * regge_factor * diff_T  

def differential_sigma(amp_value, s):
    amp_squared = amp_value.imag * amp_value.imag
    denominator = (16 * np.pi * s**2)
    return amp_squared / denominator * 0.389379323

def full_int(mg, a1, a2, m2_func, q_val, sqrt_s):

    def integrand(y, x, mg, a1, a2, m2_func, q_val):
        k        = sqrt_s * x
        phi      = 2*np.pi*y
        jacobian = 2*np.pi*sqrt_s
        return k*(T_1(k,q_val,phi,mg,a1,a2,m2_func) -
                  T_2(k,q_val,phi,mg,a1,a2,m2_func))*jacobian
    def inner_integral(x):
            return fixed_quad(
                lambda y: integrand(y, x, mg, a1, a2, m2_func, q_val),
                0, 1, n=n_points
            )[0]

    integral_value = fixed_quad(inner_integral, 0, 1, n=n_points)[0]
    
    return integral_value

In [24]:
def model_function(x, eps, mg, a1, a2, sqrt_s, model_type='log'):

    # Definindo os parâmetros específicos do modelo
    params = {
        'epsilon': eps,
        'mg': mg,
        'a1': a1,
        'a2': a2
    }
    
    # Escolhendo a massa conforme o modelo
    m2 = m2_log if model_type == 'log' else m2_pl
    
    dif_sigma_lst = []
    
    for q2 in x:
        t = -q2
        
        integral_value = full_int(mg, a1, a2, m2, q2, sqrt_s)

        diff_T = integral_value
        s = sqrt_s ** 2
        amp_value = amp_calculation(diff_T, s, params['epsilon'], t)
        dif_sigma_value = differential_sigma(amp_value, s)
        dif_sigma_lst.append(dif_sigma_value)
    
    return np.array(dif_sigma_lst)

In [25]:
def create_least_squares(x, y, yerr, sqrt_s, model_type):
    return LeastSquares(x, y, yerr, 
        lambda x, eps, mg, a1, a2: model_function(x, eps, mg, a1, a2, sqrt_s, model_type))

def total_cost(dataset, model_type):
    return sum([
        create_least_squares(*dataset[7000], 7000, model_type),
        create_least_squares(*dataset[8000], 8000, model_type),
        create_least_squares(*dataset[13000], 13000, model_type)
    ])

# Estrutura dos dados por experimento
atlas_data = {
    7000: (x_7_atlas, y_7_atlas, yerr_7_atlas),
    8000: (x_8_atlas, y_8_atlas, yerr_8_atlas),
    13000: (x_13_atlas, y_13_atlas, yerr_13_atlas)
}


# Custos totais
total_cost_pl_atlas = total_cost(atlas_data, 'pl')

In [ ]:
def otimization(total_cost_func, initial_params, model_type: str, ensemble: str):
    print('\n')
    print(80 * '-')
    print(f"Iniciando otimização dos parâmetros usando LeastSquares para {model_type} em {ensemble.upper()}")

    start_time = time.time()
    
    m = Minuit(total_cost_func, 
               eps=initial_params['epsilon'],
               mg=initial_params['mg'],
               a1=initial_params['a1'],
               a2=initial_params['a2'])
    # Configurações adicionais
    down = 0.99
    up = 2- down

    m.fixed['a1'] = True
    m.fixed['a2'] = True

    m.limits['mg'] = (down * initial_params['mg'], up *initial_params['mg'])
    m.limits['eps'] = (down * initial_params['epsilon'], up *initial_params['epsilon'])
    # m.limits['a2'] = (down * initial_params['a2'], up * initial_params['a2'])
    # m.limits['a1'] = (down * initial_params['a1'], up * initial_params['a1'])

    # Configurações adicionais
    m.strategy = 2
    # m.errordef = 7.79
    m.tol = 1e-1
    
    m.migrad(ncall = 300)
    m.fixed = False
    m.migrad(ncall=300)
    m.hesse()


    print(f'Finalizado a minimização para {model_type} em {ensemble}')

    execution_time = time.time() - start_time
    
    minutes = int(execution_time // 60)
    seconds = execution_time % 60
    print(f'Tempo de execução para {model_type} em {ensemble}: {minutes} min {seconds:.2f} s \n')
    
    print(f"Parâmetros otimizados para {model_type} em {ensemble}: \n")
    print(f'mg: {m.values["mg"]} ± {m.errors["mg"]}')
    print(f'eps: {m.values["eps"]} ± {m.errors["eps"]}')
    print(f'a1: {m.values["a1"]} ± {m.errors["a1"]}')
    print(f'a2: {m.values["a2"]} ± {m.errors["a2"]}')
    print(f'chi2/ndof: {m.fval / m.ndof}')


    return m



In [27]:
m_pl_atlas = otimization(total_cost_pl_atlas,
                         initial_params_pl_atlas,
                         'pl', 'atlas')




--------------------------------------------------------------------------------
Iniciando otimização dos parâmetros usando LeastSquares para pl em ATLAS


/home/victorli/miniconda3/envs/env_iminuit/lib/python3.13/site-packages/iminuit/cost.py:173: RuntimeWarning: overflow encountered in multiply
  return z * z
/tmp/ipykernel_7463/1414407444.py:15: RuntimeWarning: overflow encountered in exp
  return np.exp(-(a1 * q2 + a2 * q2 ** 2))
/tmp/ipykernel_7463/1414407444.py:41: RuntimeWarning: overflow encountered in multiply
  return alpha_D_plus * alpha_D_minus * G_minus * (2 * G0 - G_minus)
/tmp/ipykernel_7463/1414407444.py:68: RuntimeWarning: overflow encountered in multiply
  return k*(T_1(k,q_val,phi,mg,a1,a2,m2_func) -


Finalizado a minimização para pl em atlas
Tempo de execução para pl em atlas: 1 min 56.71 s 

Parâmetros otimizados para pl em atlas: 

mg: 0.42106985186628554 ± 0.0010902878607861877
eps: 0.0746857106471165 ± 0.001410328824399537
a1: 1.517 ± 0.03902466370419308
a2: 2.05 ± 0.20794094401865962
chi2/ndof: 1.1518133422115622


In [10]:
print(m_pl_atlas.valid)

True


In [11]:
def get_dif_sigma(epsilon, mg, a1, a2, mg_model):

    sqrt_s = 7000
    scale = 1  # caso único
    start_q2 = 0.006
    max_q2   = 0.204
    q2_step  = 0.001
    n_points = 10000

    # def integrand(y, x, mg, a1, a2, m2_func, q_val):
    #     k        = sqrt_s * x
    #     phi      = 2*np.pi*y
    #     jacobian = 2*np.pi*sqrt_s
    #     return k*(T_1(k,q_val,phi,mg,a1,a2,m2_func) -
    #               T_2(k,q_val,phi,mg,a1,a2,m2_func))*jacobian

    lst_q2 = []
    lst_dif_sigma = []

    q2 = start_q2
    while q2 <= max_q2:
        t = -q2

        integral_value = full_int(mg, a1, a2, mg_model, q2, sqrt_s)

        diff_T = integral_value

        s          = sqrt_s**2
        amp_value  = amp_calculation(diff_T, s, epsilon, t)
        dif_sigma  = differential_sigma(amp_value, s) * scale

        lst_q2.append(q2)
        lst_dif_sigma.append(dif_sigma)

        q2 += q2_step

    return {sqrt_s: (lst_q2, lst_dif_sigma)}

#for pl atlas
dif_sigma_pl_atlas = get_dif_sigma(
    m_pl_atlas.values['eps'],
    m_pl_atlas.values['mg'], 
    m_pl_atlas.values['a1'],
    m_pl_atlas.values['a2'],
    m2_pl
)

dif_sigma_pl_atlas_7_q2 = dif_sigma_pl_atlas[7000][0]
dif_sigma_pl_atlas_7_values = dif_sigma_pl_atlas[7000][1]
def add_differential_trace(fig, x, y, label, color='red', mg_model= 'log', legend=True, size = 4, width = 2):
    fig.add_trace(go.Scatter(
        x=x,
        y=y,
        mode='lines+markers',
        line=dict(color=color, width=width),
        marker=dict(size=size),
        name=f'{label}, {mg_model}',
        showlegend=legend,

    ))

def add_data_trace(fig, x, y, y_error, scale=1.0, color='black', size=4,
                           name=None, show_label=True, mode='markers'):
    fig.add_trace(go.Scatter(
        x=x,
        y=y * scale,
        mode=mode,
        marker=dict(color=color, size=size),
        error_y=dict(
            type='data',
            array=y_error * scale,
            visible=True
        ),
        name=name if show_label else None,
        showlegend=show_label
    ))

fig_atlas = go.Figure()


# for pl atlas
add_differential_trace(fig_atlas, dif_sigma_pl_atlas_7_q2, dif_sigma_pl_atlas_7_values,label='7 TeV', color='blue', mg_model='pl')

#-----------------------------------------------------------------------------------------------

#-----------------------------------------------------------------------------------------------

#data points
add_data_trace(fig_atlas, x_7_atlas, y_7_atlas, yerr_7_atlas, name='ATLAS 7 TeV', show_label=True, mode='markers')

# Atualiza layout
fig_atlas.update_layout(
    title='dσ/dt vs. |t| - Log and PL models in ATLAS',
    xaxis_title='|t| (GeV²)',
    yaxis_title='dσ/dt (mb/GeV²)',
    yaxis_type='log',
    legend_title='Mass Model',
    plot_bgcolor='white',
    hovermode='x unified'
)

fig_atlas.update_xaxes(gridcolor='lightgray')
fig_atlas.update_yaxes(gridcolor='lightgray')

# fig_atlas.show(renderer='browser')


ValueError: Mime type rendering requires nbformat>=4.2.0 but it is not installed

Figure({
    'data': [{'line': {'color': 'blue', 'width': 2},
              'marker': {'size': 4},
              'mode': 'lines+markers',
              'name': '7 TeV, pl',
              'showlegend': True,
              'type': 'scatter',
              'x': [0.006, 0.007, 0.008, 0.009000000000000001,
                    0.010000000000000002, 0.011000000000000003,
                    0.012000000000000004, 0.013000000000000005,
                    0.014000000000000005, 0.015000000000000006,
                    0.016000000000000007, 0.017000000000000008,
                    0.01800000000000001, 0.01900000000000001, 0.02000000000000001,
                    0.02100000000000001, 0.022000000000000013,
                    0.023000000000000013, 0.024000000000000014,
                    0.025000000000000015, 0.026000000000000016,
                    0.027000000000000017, 0.028000000000000018,
                    0.02900000000000002, 0.03000000000000002, 0.03100000000000002,
                    0.03200000000000002, 0.03300000000000002, 0.03400000000000002,
                    0.035000000000000024, 0.036000000000000025,
                    0.037000000000000026, 0.03800000000000003, 0.03900000000000003,
                    0.04000000000000003, 0.04100000000000003, 0.04200000000000003,
                    0.04300000000000003, 0.04400000000000003, 0.04500000000000003,
                    0.046000000000000034, 0.047000000000000035,
                    0.048000000000000036, 0.04900000000000004, 0.05000000000000004,
                    0.05100000000000004, 0.05200000000000004, 0.05300000000000004,
                    0.05400000000000004, 0.05500000000000004, 0.05600000000000004,
                    0.057000000000000044, 0.058000000000000045,
                    0.059000000000000045, 0.060000000000000046,
                    0.06100000000000005, 0.06200000000000005, 0.06300000000000004,
                    0.06400000000000004, 0.06500000000000004, 0.06600000000000004,
                    0.06700000000000005, 0.06800000000000005, 0.06900000000000005,
                    0.07000000000000005, 0.07100000000000005, 0.07200000000000005,
                    0.07300000000000005, 0.07400000000000005, 0.07500000000000005,
                    0.07600000000000005, 0.07700000000000005, 0.07800000000000006,
                    0.07900000000000006, 0.08000000000000006, 0.08100000000000006,
                    0.08200000000000006, 0.08300000000000006, 0.08400000000000006,
                    0.08500000000000006, 0.08600000000000006, 0.08700000000000006,
                    0.08800000000000006, 0.08900000000000007, 0.09000000000000007,
                    0.09100000000000007, 0.09200000000000007, 0.09300000000000007,
                    0.09400000000000007, 0.09500000000000007, 0.09600000000000007,
                    0.09700000000000007, 0.09800000000000007, 0.09900000000000007,
                    0.10000000000000007, 0.10100000000000008, 0.10200000000000008,
                    0.10300000000000008, 0.10400000000000008, 0.10500000000000008,
                    0.10600000000000008, 0.10700000000000008, 0.10800000000000008,
                    0.10900000000000008, 0.11000000000000008, 0.11100000000000008,
                    0.11200000000000009, 0.11300000000000009, 0.11400000000000009,
                    0.11500000000000009, 0.11600000000000009, 0.11700000000000009,
                    0.11800000000000009, 0.11900000000000009, 0.12000000000000009,
                    0.1210000000000001, 0.1220000000000001, 0.1230000000000001,
                    0.1240000000000001, 0.12500000000000008, 0.12600000000000008,
                    0.12700000000000009, 0.12800000000000009, 0.1290000000000001,
                    0.1300000000000001, 0.1310000000000001, 0.1320000000000001,
                    0.1330000000000001, 0.1340000000000001, 0.1350000000000001,
                    0.1360000000000001, 0.1370000000000001, 0.1380000000000001,
                    0.139000000000000

In [12]:
lst_q_integration = np.linspace(0, 0.2, 500)
lst_b_integration = np.linspace(0, 30, 500)

In [13]:
import numpy as np
from functools import lru_cache

# tenta importar j0 (Bessel) do scipy; se não estiver disponível, informa erro claro
try:
    from scipy.special import j0
except Exception as e:
    raise ImportError(
        "Necessário scipy (scipy.special.j0). Instale via `pip install scipy` ou forneça uma função j0 compatível."
    )

# ------------------------------
# --- ADAPTE/INSIRA AQUI ---
# As suas funções escalares existentes (não alteradas) devem estar disponíveis no namespace:
#    full_int(mg, a1, a2, m2_pl, q_int, sqrt_s)
#    amp_calculation(diff_t, s, epsilon, t)
#    differential_sigma(eik_amp_sum, s)
#
# Se você já tiver versões vetoriais dessas funções, remova os wrappers abaixo e use-as diretamente.
# ------------------------------

# Wrappers que aceitam arrays: se as funções originais são escalares, usamos np.vectorize.
# Se já possuir versões vetoriais, sobrescreva essas variáveis com as versões vetoriais.
def _make_vectorized_if_needed(func, nin=1, otypes=None):
    """
    Retorna uma função que aceita arrays.
    - func: função escalar original
    - nin: número de argumentos posicionais a vectorizar (others passed through)
    - otypes: opcional, tipos de saída para np.vectorize (ex: [np.complex128])
    """
    # np.vectorize preserves broadcasting semantics if inputs are arrays
    if otypes is None:
        return np.vectorize(func)
    else:
        return np.vectorize(func, otypes=otypes)

# Substitua por suas implementações vetoriais se existirem:
# Exemplo: full_int_vec = full_int_vectorizada (se você tiver)
full_int_vec = _make_vectorized_if_needed(full_int, nin=6)             # args: mg,a1,a2,m2_pl,q_int,sqrt_s
amp_calculation_vec = _make_vectorized_if_needed(amp_calculation, nin=4)  # args: diff_t, s, epsilon, t
differential_sigma_vec = _make_vectorized_if_needed(differential_sigma, nin=2)  # args: eik_amp_sum, s

# ------------------------------
# Vetorização principal - OTIMIZADA
# ------------------------------
def model_function_vectorized(x_born, eps, mg, a1, a2, sqrt_s, model_type='log',
                              lst_q_integration=None, lst_b_integration=None,
                              ensemble_parameters=None, m2_pl=None):
    """
    Versão vetorizada e otimizada de `model_function`.
    
    OTIMIZAÇÕES IMPLEMENTADAS:
    1. Cache de conversão de arrays para evitar cópias desnecessárias
    2. Pré-alocação de arrays com dtype explícito
    3. Uso de operações in-place onde possível
    4. Otimização de broadcasting com views ao invés de cópias
    5. Remoção de operações redundantes
    6. Uso de einsum para operações matriciais mais eficientes
    """

    # --- valida e padroniza inputs ---
    x_born = np.atleast_1d(x_born)
    s = sqrt_s ** 2

    if lst_q_integration is None:
        raise ValueError("lst_q_integration (lista/array de pontos de integração em q) é obrigatória")
    if lst_b_integration is None:
        raise ValueError("lst_b_integration (lista/array de pontos de integração em b) é obrigatória")
    if ensemble_parameters is None:
        raise ValueError("ensemble_parameters (dict) é obrigatório")
    if m2_pl is None:
        raise ValueError("m2_pl (parâmetro) é obrigatório")

    # OTIMIZAÇÃO 1: Usar ascontiguousarray para garantir layout de memória eficiente
    q_arr = np.ascontiguousarray(lst_q_integration, dtype=np.float64)  # shape (nq,)
    b_arr = np.ascontiguousarray(lst_b_integration, dtype=np.float64)  # shape (nb,)
    
    nq = q_arr.shape[0]
    nb = b_arr.shape[0]
    nx = x_born.shape[0]

    # --- 1) pré-calcula integrals e born amplitudes que só dependem de q_int ---
    diff_t_q = full_int_vec(mg, a1, a2, m2_pl, q_arr, sqrt_s)  # shape (nq,)

    # OTIMIZAÇÃO 2: Cálculo in-place para t_q
    t_q = np.empty(nq, dtype=np.float64)
    np.negative(q_arr, out=t_q)
    np.square(t_q, out=t_q)  # t_q = -(q^2)

    epsilon_used = ensemble_parameters['atlas']['pl']['epsilon']
    born_amp_q = amp_calculation_vec(diff_t_q, s, epsilon_used, t_q)  # shape (nq,)

    # --- 2) calcula chi_sum para cada b ---
    # OTIMIZAÇÃO 3: Pré-calcular b*q de forma eficiente usando outer product
    bq_matrix = np.outer(b_arr, q_arr)  # (nb, nq) - mais eficiente que broadcasting manual
    
    # Calcular j0 uma vez
    j0_bq = j0(bq_matrix)  # (nb, nq)
    
    # OTIMIZAÇÃO 4: Usar einsum para operação matricial mais eficiente
    # Equivalente a: term_matrix = (q_arr * j0_bq) * born_amp_q / s
    # einsum é mais rápido para operações combinadas
    chi_sum_b = np.einsum('j,ij,j->', q_arr, j0_bq, born_amp_q) / s
    
    # Correção: chi_sum_b deve ser por b, não um escalar
    # OTIMIZAÇÃO 5: Combinar multiplicações em uma operação
    q_born = q_arr * born_amp_q  # (nq,)
    chi_sum_b = np.dot(j0_bq, q_born) / s  # (nb,) - dot product é otimizado

    # --- 3) fator eikonal por b ---
    # OTIMIZAÇÃO 6: Cálculo in-place do exponencial complexo
    chi_sum_b_complex = chi_sum_b * 1j
    factor_b = np.empty(nb, dtype=np.complex128)
    np.exp(chi_sum_b_complex, out=factor_b)
    factor_b *= -1
    factor_b += 1  # factor_b = 1 - exp(1j * chi_sum_b)

    # --- 4) cálculo do eik_amp_sum para cada x_born ---
    # OTIMIZAÇÃO 7: Reusar padrão de cálculo com outer product
    bx_matrix = np.outer(b_arr, x_born)  # (nb, nx)
    j0_bx = j0(bx_matrix)  # (nb, nx)

    # OTIMIZAÇÃO 8: Combinar multiplicações escalares
    s_complex = 1j * s
    
    # Broadcast b_arr e factor_b eficientemente
    # eik_amp_bx[i,j] = b[i] * j0_bx[i,j] * factor_b[i] * (1j*s)
    b_factor = b_arr * factor_b * s_complex  # (nb,) - pré-combinar fatores
    eik_amp_bx = j0_bx * b_factor[:, np.newaxis]  # (nb, nx)

    # OTIMIZAÇÃO 9: axis=0 sum é otimizada para arrays contíguos
    eik_amp_sum_x = np.sum(eik_amp_bx, axis=0)  # (nx,)

    # --- 5) differential_sigma para cada eik_amp_sum_x ---
    diff_sigma_x = differential_sigma_vec(eik_amp_sum_x, s)  # (nx,)

    return np.asarray(diff_sigma_x)

In [14]:
def create_least_squares(x, y, yerr, sqrt_s, model_type,
                         lst_q_integration, lst_b_integration,
                         ensemble_parameters, m2_pl):

    return LeastSquares(
        x, y, yerr,
        lambda x, eps, mg, a1, a2: model_function_vectorized(
            x, eps, mg, a1, a2, sqrt_s, model_type,
            lst_q_integration, lst_b_integration,
            ensemble_parameters, m2_pl
        )
    )


def total_cost(dataset, model_type,
               lst_q_integration, lst_b_integration,
               ensemble_parameters, m2_pl):
    
    # CORRIGIDO: iterar sobre as chaves e somar os custos
    cost_objects = []
    for sqrt_s_key, (x, y, yerr) in dataset.items():
        cost = create_least_squares(
            x, y, yerr, sqrt_s_key, model_type,
            lst_q_integration, lst_b_integration,
            ensemble_parameters, m2_pl
        )
        cost_objects.append(cost)
    
    return sum(cost_objects)


# Estrutura dos dados por experimento
atlas_data = {
    7000: (x_7_atlas, y_7_atlas, yerr_7_atlas)
}


# Custos totais
total_cost_pl_atlas_eik = total_cost(
    atlas_data, 'pl',
    lst_q_integration, lst_b_integration,
    ensemble_parameters, m2_pl
)

In [15]:
def otimization(total_cost_func, initial_params, model_type: str, ensemble: str):
    print('\n')
    print(80 * '-')
    print(f"Iniciando otimização dos parâmetros usando LeastSquares para {model_type} em {ensemble.upper()}")

    start_time = time.time()
    
    m = Minuit(total_cost_func, 
               eps=initial_params['epsilon'],
               mg=initial_params['mg'],
               a1=initial_params['a1'],
               a2=initial_params['a2'])
    # Configurações adicionais
    down = 0.8
    up = 2

    m.fixed['mg'] = True
    m.fixed['a1'] = True

    # m.limits['mg'] = (down * initial_params['mg'], up *initial_params['mg'])
    # m.limits['eps'] = (down * initial_params['epsilon'], up *initial_params['epsilon'])
    # m.limits['a2'] = (down * initial_params['a2'], up * initial_params['a2'])
    # m.limits['a1'] = (down * initial_params['a1'], up * initial_params['a1'])


    
    # Configurações adicionais
    m.strategy = 2
    # m.errordef = 7.79
    # m.tol = 1e-1

    m.simplex()
    m.simplex()

    m.migrad()
    m.migrad()

    m.fixed = False

    m.limits['mg'] = (0, up *initial_params['mg'])
    m.limits['eps'] = (0, up *initial_params['epsilon'])
    m.limits['a2'] = (0, up * initial_params['a2'])
    m.limits['a1'] = (0, up * initial_params['a1'])

    m.migrad()
    m.migrad()
    m.hesse()


    print(f'Finalizado a minimização para {model_type} em {ensemble}')

    execution_time = time.time() - start_time
    
    minutes = int(execution_time // 60)
    seconds = execution_time % 60
    print(f'Tempo de execução para {model_type} em {ensemble}: {minutes} min {seconds:.2f} s \n')
    
    print(f"Parâmetros otimizados para {model_type} em {ensemble}: \n")
    print(f'mg: {m.values["mg"]} ± {m.errors["mg"]}')
    print(f'eps: {m.values["eps"]} ± {m.errors["eps"]}')
    print(f'a1: {m.values["a1"]} ± {m.errors["a1"]}')
    print(f'a2: {m.values["a2"]} ± {m.errors["a2"]}')
    print(f'chi2/ndof: {m.fval / m.ndof}')


    return m



In [16]:
m_pl_atlas = otimization(total_cost_pl_atlas_eik,
                         initial_params_pl_atlas,
                         'pl', 'atlas')




--------------------------------------------------------------------------------
Iniciando otimização dos parâmetros usando LeastSquares para pl em ATLAS


/tmp/ipykernel_31648/1414407444.py:58: RuntimeWarning:

overflow encountered in scalar multiply

/home/victorli/miniconda3/envs/env_iminuit/lib/python3.13/site-packages/numpy/lib/_function_base_impl.py:2625: RuntimeWarning:

overflow encountered in differential_sigma (vectorized)

/home/victorli/miniconda3/envs/env_iminuit/lib/python3.13/site-packages/iminuit/cost.py:173: RuntimeWarning:

overflow encountered in multiply

/tmp/ipykernel_31648/3883934811.py:114: RuntimeWarning:

overflow encountered in exp

/tmp/ipykernel_31648/3883934811.py:115: RuntimeWarning:

invalid value encountered in multiply

/tmp/ipykernel_31648/3883934811.py:128: RuntimeWarning:

invalid value encountered in multiply

/tmp/ipykernel_31648/3883934811.py:128: RuntimeWarning:

overflow encountered in multiply

/tmp/ipykernel_31648/3883934811.py:129: RuntimeWarning:

invalid value encountered in multiply



Finalizado a minimização para pl em atlas
Tempo de execução para pl em atlas: 46 min 22.35 s 

Parâmetros otimizados para pl em atlas: 

mg: 0.421 ± nan
eps: 0.0753 ± nan
a1: 1.517 ± nan
a2: 2.05 ± nan
chi2/ndof: inf


In [17]:
print(m_pl_atlas.valid)
# mg: 0.8419991970064808 ± 0.0
# eps: 0.14316961094784592 ± 0.0
# a1: 3.0339971065530436 ± 0.0
# a2: 4.099996089936545 ± 0.0
# chi2/ndof: 25004992576.01236

False
